# 01. 법령 API 데이터 수집

법제처 **국가법령정보 공동활용 API**(law.go.kr DRF)로 법령·판례·법령해석례를 수집해
`data/01_raw/`에 저장한다.

| target  | 설명        | 검색 방식        | 중복 제거 기준       | 저장 형식                                  |
| ------- | ----------- | ---------------- | -------------------- | ------------------------------------------ |
| `eflaw` | 현행법령    | 명칭검색(search=1) | `법령ID`             | JSON + 본문 MD `api_eflaw/md/{법령명}.md` |
| `prec`  | 판례        | 본문검색(search=2) | `판례일련번호`       | JSONL + 본문 MD `api_prec/md/{ID}.md`     |
| `expc`  | 법령해석례  | 본문검색(search=2) | `법령해석례일련번호` | JSONL + 본문 MD `api_expc/md/{ID}.md`     |

처리 흐름: **검색 → (카테고리 태깅) → 합치기 → 1차 dedup(ID) → 상세 조회 → 2차 검증 → JSON/JSONL 및 본문 MD 저장**

> ⚠️ 2차 검증에서 본문이 비었거나 `"일치하는 …없습니다"` 인 레코드를 제거한다.
> ⚠️ `상세링크` 필드에는 OC(API 키)가 노출되므로 저장 전에 제거한다.
> 💡 각 타깃(eflaw/prec/expc)은 서로 독립 셀이며, 공통 **설정 셀 + 헬퍼 셀**만 먼저 실행돼 있으면 개별 단독 실행 가능하다.

## 0. 공통 설정

In [ ]:
import json
import math
import re
import time
from pathlib import Path
from typing import Any, Literal

import requests
from dotenv import load_dotenv
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import os

load_dotenv("../.env")
OC = os.environ["LAW_API_OC"]  # .env 에 LAW_API_OC 필요

SEARCH_URL = "https://www.law.go.kr/DRF/lawSearch.do"
SERVICE_URL = "https://www.law.go.kr/DRF/lawService.do"

# 재시도(429/5xx) + 백오프 세션
_retry = Retry(total=3, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504])
_session = requests.Session()
_session.mount("https://", HTTPAdapter(max_retries=_retry))
_session.mount("http://", HTTPAdapter(max_retries=_retry))

# 저장 경로
RAW_DIR = Path("../data/01_raw")
EFLAW_DIR = RAW_DIR / "api_eflaw"
EFLAW_DIR.mkdir(parents=True, exist_ok=True)

print("LAW_API_OC 설정: 완료")
print("저장 경로:", RAW_DIR.resolve())

### API 호출 헬퍼

- `api()` : 단일 호출 래퍼 (`resultCode` 검사 포함)
- `fetch_list()` : 목록 조회 + 페이지네이션 자동 처리
- `fetch_details()` : 항목별 상세 조회 + 본문 유효성 필터 + 메타 병합

In [ ]:
# 검색/상세 응답의 루트·items 키가 target 마다 달라 헬퍼로 흡수한다.
_SEARCH_ROOT = {"eflaw": "LawSearch", "prec": "PrecSearch", "expc": "Expc"}
_SEARCH_ITEMS = {"eflaw": "law", "prec": "prec", "expc": "expc"}


def api(
    target: str,
    service: Literal["search", "detail"] = "search",
    params: dict[str, Any] | None = None,
    timeout: int = 60,
) -> dict[str, Any]:
    """law.go.kr DRF 단일 호출. JSON dict 반환."""
    url = SERVICE_URL if service == "detail" else SEARCH_URL
    req = {"OC": OC, "type": "JSON", "target": target, **(params or {})}
    resp = _session.get(url, params=req, timeout=timeout)
    resp.raise_for_status()
    return resp.json()


def _search_root(resp: dict, target: str) -> dict:
    return resp.get(_SEARCH_ROOT[target], {})


def _extract_items(resp: dict, target: str) -> list[dict]:
    root = _search_root(resp, target)
    items = root.get(_SEARCH_ITEMS[target], [])
    if isinstance(items, dict):  # 결과 1건이면 dict 로 옴
        items = [items]
    return items if isinstance(items, list) else []


def fetch_list(
    target: str,
    query: str,
    search: int = 1,
    max_items: int | None = None,
    extra: dict[str, Any] | None = None,
) -> list[dict]:
    """목록 조회 + 페이지네이션. search=1 명칭 / search=2 본문.
    extra: target별 추가 파라미터(예: eflaw 의 nw=3 → 현행만)."""
    display = 100
    base = {"query": query, "search": search, "display": display, **(extra or {})}

    first = api(target, "search", {**base, "page": 1})
    total = int(_search_root(first, target).get("totalCnt", 0))
    if max_items:
        total = min(total, max_items)
    pages = math.ceil(total / display)

    items = _extract_items(first, target)
    for page in range(2, pages + 1):
        items.extend(_extract_items(api(target, "search", {**base, "page": page}), target))
        time.sleep(0.3)
    return items[:max_items] if max_items else items


def _strip_link_fields(item: dict) -> None:
    """상세링크 계열 필드 제거 (OC=API키 노출 방지)."""
    for k in [k for k in item if "상세링크" in k]:
        item.pop(k, None)


def _is_invalid_body(body: Any) -> bool:
    """빈 본문 또는 '일치하는 …없습니다' 에러 응답이면 True."""
    if not body:
        return True
    s = json.dumps(body, ensure_ascii=False)
    return ("일치하는" in s and "없습니다" in s)


def fetch_details(
    target: str,
    items: list[dict],
    id_field: str,
    delay: float = 0.5,
) -> list[dict]:
    """항목별 상세 조회 → 본문 유효성 필터 → 메타+본문 병합."""
    results, dropped = [], 0
    total = len(items)
    print(f"  상세 조회 시작: 총 {total}건 (건당 ~{delay}s sleep)")
    for i, item in enumerate(items, 1):
        doc_id = str(item.get(id_field, ""))
        if not doc_id:
            dropped += 1
            continue
        body = api(target, "detail", {"ID": doc_id})
        time.sleep(delay)
        if _is_invalid_body(body):
            dropped += 1
            continue
        _strip_link_fields(item)
        results.append({**item, "본문": body})
        if i % 20 == 0 or i == total:
            print(f"    진행 {i}/{total} — 유효 {len(results)} / 제거 {dropped}", flush=True)
    print(f"  상세 조회: {len(results)}건 유효 / {dropped}건 제거(빈·에러 본문)")
    return results


def dedup_by(items: list[dict], id_field: str) -> list[dict]:
    """id_field 기준 1차 중복 제거(빈 ID 제외)."""
    seen: set[str] = set()
    out: list[dict] = []
    for item in items:
        doc_id = str(item.get(id_field, ""))
        if doc_id and doc_id not in seen:
            seen.add(doc_id)
            out.append(item)
    return out

# ── 병렬 상세 조회 지원 ──────────────────────────────────────────────────────────
import threading
import collections
from concurrent.futures import ThreadPoolExecutor, as_completed


class RateLimiter:
    """슬라이딩 윈도우 방식으로 초당 max_rps 요청 제한."""

    def __init__(self, max_rps: float = 5.0) -> None:
        self._lock = threading.Lock()
        self._calls: collections.deque = collections.deque()
        self._max_rps = max_rps
        self._window = 1.0

    def wait(self) -> None:
        with self._lock:
            now = time.monotonic()
            while self._calls and self._calls[0] < now - self._window:
                self._calls.popleft()
            if len(self._calls) >= self._max_rps:
                sleep_until = self._calls[0] + self._window
                time.sleep(max(0, sleep_until - now))
            self._calls.append(time.monotonic())


_rate_limiter = RateLimiter(max_rps=2.0)


def fetch_details_parallel(
    target: str,
    items: list[dict],
    id_field: str,
    checkpoint_path: Path | None = None,
    max_workers: int = 5,
) -> list[dict]:
    """병렬 상세 조회 + 즉시 체크포인트 저장 + resume 지원."""
    done_ids: set[str] = set()
    if checkpoint_path and checkpoint_path.exists():
        with checkpoint_path.open(encoding="utf-8") as f:
            for line in f:
                rec = json.loads(line)
                doc_id = str(rec.get(id_field, ""))
                if doc_id:
                    done_ids.add(doc_id)
        print(f"  resume: {len(done_ids)}건 이미 완료 → 스킵")

    pending = [it for it in items if str(it.get(id_field, "")) not in done_ids]
    total = len(pending)
    print(f"  병렬 상세 조회: {total}건 남음 (workers={max_workers})")

    file_lock = threading.Lock()
    dropped = 0
    done_count = len(done_ids)

    def _fetch_one(item: dict) -> dict | None:
        doc_id = str(item.get(id_field, ""))
        if not doc_id:
            return None
        DETAIL_MAX_ATTEMPTS = 4
        for attempt in range(DETAIL_MAX_ATTEMPTS):
            _rate_limiter.wait()
            body = api(target, "detail", {"ID": doc_id})
            if not _is_invalid_body(body):
                _strip_link_fields(item)
                return {**item, "본문": body}
            if attempt + 1 < DETAIL_MAX_ATTEMPTS:
                time.sleep(1.5 * (attempt + 1))
        return None

    ck_file = checkpoint_path.open("a", encoding="utf-8") if checkpoint_path else None
    try:
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {executor.submit(_fetch_one, it): it for it in pending}
            for i, fut in enumerate(as_completed(futures), 1):
                rec = fut.result()
                if rec is None:
                    dropped += 1
                else:
                    with file_lock:
                        if ck_file:
                            ck_file.write(json.dumps(rec, ensure_ascii=False) + "\n")
                            ck_file.flush()
                    done_count += 1
                if i % 50 == 0 or i == total:
                    print(f"    진행 {i}/{total} — 유효 {done_count} / 제거 {dropped}", flush=True)
    finally:
        if ck_file:
            ck_file.close()

    if checkpoint_path and checkpoint_path.exists():
        with checkpoint_path.open(encoding="utf-8") as f:
            return [json.loads(line) for line in f if line.strip()]
    return []

def dedup_with_provenance(items: list[dict], id_field: str) -> list[dict]:
    """ID 중복은 제거하되 매칭된 카테고리와 검색어는 모두 보존한다."""
    merged: dict[str, dict] = {}
    order: list[str] = []

    for item in items:
        item_id = str(item.get(id_field, "")).strip()
        if not item_id:
            continue

        categories = list(item.get("_categories") or [])
        queries = list(item.get("_queries") or [])
        if item.get("_category"):
            categories.append(item["_category"])
        if item.get("_query"):
            queries.append(item["_query"])

        if item_id not in merged:
            merged[item_id] = dict(item)
            merged[item_id]["_categories"] = []
            merged[item_id]["_queries"] = []
            order.append(item_id)

        record = merged[item_id]
        for value in categories:
            if value and value not in record["_categories"]:
                record["_categories"].append(value)
        for value in queries:
            if value and value not in record["_queries"]:
                record["_queries"].append(value)

        record["_category"] = record["_categories"][0] if record["_categories"] else "기타"
        record["_query"] = record["_queries"][0] if record["_queries"] else ""

    return [merged[item_id] for item_id in order]

# API 상세 응답을 JSON 구조 없이 읽기 좋은 본문 Markdown으로 함께 저장한다.
import html


_API_TEXT_FIELD_ORDER = {
    "eflaw": (
        "법령명한글", "법령명_한글", "개정문내용", "제개정이유내용",
        "조문내용", "항내용", "호내용", "목내용", "부칙내용", "별표내용",
    ),
    "prec": ("사건명", "판시사항", "판결요지", "참조조문", "참조판례", "판례내용"),
    "expc": ("안건명", "질의요지", "회답", "이유"),
}
_TABLE_LAYOUT_ONLY_RE = re.compile(
    r"^[\s│┃║─━═┄┅┈┉╌╍┌┐└┘├┤┬┴┼┏┓┗┛┣┫┳┻╋╔╗╚╝╠╣╦╩╬|+_-]+$"
)
_KOREAN_GUIDE_RE = re.compile(r"작성\s*방법\s*\(")
_OFFICIAL_ENGLISH_ANNEX_RE = re.compile(
    r"(?im)^[ \t]*■\s*Enforcement\s+Rules?\s+of\s+"
    r"Licensed\s+Real\s+Estate\s+Agents?\s+Act\b"
)
_HTML_TAG_RE = re.compile(
    r"</?(?:p|div|span|strong|b|i|u|table|thead|tbody|tr|td|th|ul|ol|li)\b[^>]*>",
    re.IGNORECASE,
)


def _flatten_api_value(value) -> str:
    if value is None:
        return ""
    if isinstance(value, (list, tuple)):
        return "\n".join(
            text for item in value if (text := _flatten_api_value(item))
        )
    if isinstance(value, dict):
        return "\n".join(
            text for item in value.values() if (text := _flatten_api_value(item))
        )
    return str(value)


def _remove_duplicate_official_english(text: str) -> str:
    korean_guide = _KOREAN_GUIDE_RE.search(text)
    if not korean_guide:
        return text
    english_annex = _OFFICIAL_ENGLISH_ANNEX_RE.search(text, korean_guide.end())
    return text[:english_annex.start()] if english_annex else text


def _clean_api_text(value) -> str:
    text = html.unescape(_remove_duplicate_official_english(_flatten_api_value(value)))
    text = re.sub(r"(?i)<br\s*/?>", "\n", text)
    text = _HTML_TAG_RE.sub(" ", text).replace("\u00a0", " ")
    lines = []
    for raw_line in text.replace("\r\n", "\n").replace("\r", "\n").splitlines():
        line = re.sub(r"[ \t]+", " ", raw_line).strip()
        if not line or _TABLE_LAYOUT_ONLY_RE.fullmatch(line):
            continue
        line = re.sub(r"[│┃║]", " | ", line)
        line = re.sub(r"[─━═┄┅┈┉╌╍]+", " ", line)
        line = re.sub(r"[┌┐└┘├┤┬┴┼┏┓┗┛┣┫┳┻╋╔╗╚╝╠╣╦╩╬]", " ", line)
        line = re.sub(r"\s*\|\s*", " | ", line)
        line = re.sub(r"[ \t]+", " ", line).strip(" |")
        if line:
            lines.append(line)
    return "\n".join(lines).strip()


def api_response_to_text(target: str, response: dict) -> str:
    """타깃별 본문 필드만 API 응답 순서대로 모은다."""
    field_order = _API_TEXT_FIELD_ORDER[target]
    fields = set(field_order)
    parts: list[str] = []

    def append_text(value) -> None:
        text = _clean_api_text(value)
        if text and (not parts or parts[-1] != text):
            parts.append(text)

    def collect_in_response_order(value) -> None:
        if isinstance(value, dict):
            for key, nested in value.items():
                if key in fields:
                    append_text(nested)
                else:
                    collect_in_response_order(nested)
        elif isinstance(value, (list, tuple)):
            for nested in value:
                collect_in_response_order(nested)

    def collect_field(value, wanted: str) -> None:
        if isinstance(value, dict):
            for key, nested in value.items():
                if key == wanted:
                    append_text(nested)
                else:
                    collect_field(nested, wanted)
        elif isinstance(value, (list, tuple)):
            for nested in value:
                collect_field(nested, wanted)

    if target == "eflaw":
        # 제목은 첫 줄에 두고, 조문-항-호-목은 API의 원래 중첩 순서를 보존한다.
        for field in ("법령명한글", "법령명_한글"):
            collect_field(response, field)
        fields.difference_update({"법령명한글", "법령명_한글"})
        collect_in_response_order(response)
    else:
        # 판례·해석례는 제목부터 시작하도록 사람이 읽는 순서를 고정한다.
        for field in field_order:
            collect_field(response, field)
    return "\n\n".join(parts).strip()


def api_response_to_markdown(target: str, response: dict) -> str:
    """본문 첫 줄을 문서 제목으로 사용한 metadata 없는 Markdown을 만든다."""
    text = api_response_to_text(target, response)
    if not text:
        return ""
    title, separator, body = text.partition("\n")
    return f"# {title}\n" + (f"\n{body.lstrip()}" if separator else "")


def write_api_markdown_files(
    target: str,
    records: list[dict],
    output_dir: Path,
    filename_field: str,
) -> int:
    """상세 API 응답의 본문만 문서별 UTF-8 Markdown으로 저장한다."""
    output_dir.mkdir(parents=True, exist_ok=True)
    written = 0
    for record in records:
        markdown = api_response_to_markdown(target, record.get("본문", {}))
        if not markdown:
            continue
        stem = re.sub(r'[\\/:*?"<>|]+', "_", str(record.get(filename_field) or "unknown"))
        (output_dir / f"{stem}.md").write_text(markdown + "\n", encoding="utf-8")
        written += 1
    return written


## 1. 법령 (eflaw, 현행법령)

정확한 법령명으로 명칭검색(`search=1`, `nw=3`=현행만) → `법령명한글` 정확 일치 + `현행연혁코드=="현행"`
만 채택 → `법령ID` dedup → 상세 조회 → **법령별 개별 JSON** 으로 저장.

> ⚠️ eflaw 도 `nw` 없이 검색하면 같은 `법령ID` 의 **연혁(과거 시행본)이 함께** 반환된다.
> (예: 주택임대차보호법 → 29건) `nw=3` 으로 현행만 받아 법명당 1건이 곧 최신 현행이 되게 한다.

In [ ]:
EFLAW_QUERIES: list[str] = [
    # 기본 6
    "주택임대차보호법",
    "주택임대차보호법 시행령",
    "민법",
    "부동산등기법",
    "공인중개사법",
    "민사집행법",
    # 추가 7
    "전세사기피해자 지원 및 주거안정에 관한 특별법",
    "부동산 거래신고 등에 관한 법률",
    "주민등록법",
    "상가건물 임대차보호법",
    "부동산등기규칙",
    "공인중개사법 시행규칙",
    # 확장 4
    "민간임대주택에 관한 특별법",
    "국세징수법",
    "집합건물의 소유 및 관리에 관한 법률",
    "주택도시기금법",
]


def _safe_filename(name: str) -> str:
    """법령명 → 안전한 파일명(공백·특수문자 정리)."""
    name = re.sub(r'[\\/:*?"<>|]', "", name)
    return re.sub(r"\s+", "_", name.strip())


# 1) 명칭검색(nw=3=현행만) → 정확 일치 + 현행 필터 → 법령ID dedup
#    ※ eflaw 도 nw 없이는 연혁(과거 시행본)이 함께 반환되므로 nw=3 으로 현행만 받는다.
eflaw_items: list[dict] = []
for q in EFLAW_QUERIES:
    hits = fetch_list("eflaw", query=q, search=1, extra={"nw": 3})
    exact = [
        it for it in hits
        if it.get("법령명한글", "").strip() == q and it.get("현행연혁코드") == "현행"
    ]
    print(f"[{q}] 검색 {len(hits)}건 → 정확일치·현행 {len(exact)}건")
    eflaw_items.extend(exact)
    time.sleep(0.3)

eflaw_items = dedup_by(eflaw_items, "법령ID")
print(f"\n법령ID dedup 후: {len(eflaw_items)}건")
missing = [q for q in EFLAW_QUERIES if q not in {it.get("법령명한글", "").strip() for it in eflaw_items}]
if missing:
    print("⚠️ 정확 일치 못 찾은 query(정식 명칭 확인 필요):", missing)

In [ ]:
# 2) 상세 조회(법령ID) → 본문 유효성 필터 → 2차 dedup
eflaw_details = fetch_details("eflaw", eflaw_items, id_field="법령ID")
eflaw_details = dedup_by(eflaw_details, "법령ID")

# 3) 법령별 개별 JSON 저장
for rec in eflaw_details:
    fname = _safe_filename(rec.get("법령명한글", rec.get("법령ID", "unknown")))
    path = EFLAW_DIR / f"{fname}.json"
    with path.open("w", encoding="utf-8") as f:
        json.dump(rec, f, ensure_ascii=False, indent=2)

eflaw_md_count = write_api_markdown_files(
    "eflaw", eflaw_details, EFLAW_DIR / "md", "법령명한글"
)
print(f"법령 저장 완료: JSON {len(eflaw_details)}개 + MD {eflaw_md_count}개 → {EFLAW_DIR}")
for rec in eflaw_details:
    print(f"  - {rec.get('법령명한글')} (시행 {rec.get('시행일자')}, 법령ID {rec.get('법령ID')})")

## 2. 판례 (prec)

쟁점 키워드로 본문검색(`search=2`) → `판례일련번호` 1차 dedup → 상세 조회 →
**2차 검증(본문 유효성 + ID 유일성)** → `prec.jsonl` 저장.
단일어는 노이즈가 커서 2어절 쟁점 구 위주로 구성.

In [ ]:
# 넓은 단일어보다 임대차 문맥이 포함된 검색어를 사용한다.
PREC_QUERY_CONFIG: list[dict] = [
    {"query": "임대차보증금", "category": "보증금권리"},
    {"query": "임대차보증금 반환", "category": "보증금권리"},
    {"query": "임차인 대항력", "category": "보증금권리"},
    {"query": "임차인 우선변제권", "category": "보증금권리"},
    {"query": "소액임차인 최우선변제", "category": "보증금권리"},
    {"query": "임대차 확정일자", "category": "보증금권리"},
    {"query": "임차권등기명령", "category": "보증금권리"},
    {"query": "임차인 전입신고", "category": "보증금권리"},
    {"query": "계약갱신청구권", "category": "갱신종료"},
    {"query": "임대차 묵시적 갱신", "category": "갱신종료"},
    {"query": "임대차 갱신거절", "category": "갱신종료"},
    {"query": "임대차 해지", "category": "갱신종료"},
    {"query": "임차인 차임 연체", "category": "갱신종료"},
    {"query": "임대차 차임 증감", "category": "갱신종료"},
    {"query": "전세사기", "category": "전세사기"},
    {"query": "임대차 가장임대차", "category": "전세사기"},
    {"query": "임대차 무권대리", "category": "전세사기"},
    {"query": "임대차 이중계약", "category": "전세사기"},
    {"query": "임대차보증금 사해행위", "category": "전세사기"},
    {"query": "임대차보증금 명의신탁", "category": "전세사기"},
    {"query": "임차인 임의경매", "category": "경매배당"},
    {"query": "임차인 강제경매", "category": "경매배당"},
    {"query": "임차인 배당요구", "category": "경매배당"},
    {"query": "임차인 배당이의", "category": "경매배당"},
    {"query": "임차인 인도명령", "category": "경매배당"},
    {"query": "임대차 건물명도", "category": "경매배당"},
    {"query": "임대인 수선의무", "category": "수선원상회복"},
    {"query": "임대차 원상회복", "category": "수선원상회복"},
    {"query": "임대차 누수", "category": "수선원상회복"},
    {"query": "임대차 하자", "category": "수선원상회복"},
    {"query": "임대차 통상의 손모", "category": "수선원상회복"},
    {"query": "공인중개사 책임", "category": "중개"},
    {"query": "중개대상물 확인설명", "category": "중개"},
    {"query": "부동산 중개보수", "category": "중개"},
]

QUERY_CATEGORY = {item["query"]: item["category"] for item in PREC_QUERY_CONFIG}
PREC_QUERIES = [item["query"] for item in PREC_QUERY_CONFIG]
MAX_PER_QUERY = 300

# 본문검색 결과는 후보군이며 상세 본문에서 관련성을 다시 판정한다.
prec_items: list[dict] = []
for spec in PREC_QUERY_CONFIG:
    query = spec["query"]
    hits = fetch_list("prec", query=query, search=2, max_items=MAX_PER_QUERY)
    for hit in hits:
        hit["_category"] = spec["category"]
        hit["_query"] = query
    print(f"[{query}] {len(hits)}건")
    prec_items.extend(hits)
    time.sleep(0.3)

before = len(prec_items)
prec_items = dedup_with_provenance(prec_items, "판례일련번호")
print(f"\n수집 {before}건 → 판례일련번호 중복 제거 {len(prec_items)}건")


In [ ]:
from collections import defaultdict

# 저장 경로: 타깃별 하위폴더 (api_eflaw/, api_expc/ 와 동일 구조)
PREC_DIR = RAW_DIR / "api_prec"
PREC_DIR.mkdir(parents=True, exist_ok=True)

# 2) 병렬 상세 조회 → 체크포인트 즉시 저장 → 2차 dedup
PREC_CHECKPOINT = PREC_DIR / "prec_checkpoint.jsonl"
prec_details = fetch_details_parallel(
    "prec", prec_items, id_field="판례일련번호",
    checkpoint_path=PREC_CHECKPOINT,
    max_workers=5,
)
prec_details = dedup_with_provenance(prec_details, "판례일련번호")

# 사건번호 중복(같은 사건 다른 심급)은 로깅만, 제거는 전처리 단계 판단
case_nos = [str(r.get("사건번호", "")) for r in prec_details if r.get("사건번호")]
dup_cases = len(case_nos) - len(set(case_nos))
print(f"판례 최종 {len(prec_details)}건 (사건번호 중복 {dup_cases}건 — 전처리에서 판단)")

# 3) 카테고리별 JSONL 저장
by_cat: dict[str, list] = defaultdict(list)
for rec in prec_details:
    by_cat[rec.get("_category", "기타")].append(rec)

for cat, recs in sorted(by_cat.items()):
    cat_path = PREC_DIR / f"prec_{cat}.jsonl"
    with cat_path.open("w", encoding="utf-8") as f:
        for rec in recs:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print(f"저장 → {cat_path.name}  ({len(recs)}건)")

prec_md_count = write_api_markdown_files(
    "prec", prec_details, PREC_DIR / "md", "판례일련번호"
)
print(f"MD 저장 → {PREC_DIR / 'md'}  ({prec_md_count}개)")
print(f"\n체크포인트 유지 → {PREC_CHECKPOINT}  (재실행 시 resume 용)")


### 2-1. 진단: 판례 쿼리별 실제 가능 건수(totalCnt)

`MAX_PER_QUERY = 300` 은 **코드가 임의로 건 상한**이지 API 한계가 아니다.
판례 API 문서상 `display` 최대 100/페이지, page·총건수 상한은 **명시 없음**.
정렬은 `sort` 미지정 시 기본값 `ddes`(**선고일자 내림차순 = 최신순**) — 즉 300 cap 은 각 쿼리의 **최신 300건**을 남긴 것.

아래 셀은 쿼리별 실제 `totalCnt` 와 300 cap 손실량을 확인한다.
(설정 셀·헬퍼 셀만 있으면 prec 수집 셀 없이 **단독 실행** 가능.)

In [ ]:
# 진단: prec 쿼리별 totalCnt(실제 가능 건수) vs 300 cap
# PREC_QUERIES 가 아직 정의 안 됐으면(수집 셀 미실행) 여기서 자체 정의
PREC_QUERIES = globals().get("PREC_QUERIES") or [
    "임대차보증금", "보증금 반환", "대항력", "우선변제권", "최우선변제",
    "소액임차인", "확정일자", "임차권등기명령", "전입신고",
    "계약갱신청구권", "묵시적 갱신", "갱신거절", "임대차 해지", "차임 연체", "차임 증감",
    "전세사기", "깡통전세", "가장임대차", "무권대리", "이중임대차", "사해행위", "명의신탁",
    "임의경매", "강제경매", "배당요구", "배당이의", "인도명령", "건물명도",
    "임대인 수선의무", "원상회복", "누수", "하자", "통상의 손모",
    "공인중개사 책임", "중개대상물 확인설명", "중개보수",
]

CAP = 300
print(f"{'쿼리':<18}{'totalCnt':>10}{'수집(cap300)':>13}{'초과':>8}")
print("-" * 49)
total_all = capped = 0
for q in PREC_QUERIES:
    first = api("prec", "search", {"query": q, "search": 2, "display": 1, "page": 1})
    total = int(_search_root(first, "prec").get("totalCnt", 0))
    over = max(0, total - CAP)
    total_all += total
    capped += min(total, CAP)
    flag = f"+{over}" if over else ""
    print(f"{q:<18}{total:>10}{min(total, CAP):>13}{flag:>8}")
    time.sleep(0.3)

print("-" * 49)
print(f"{'합계':<18}{total_all:>10}{capped:>13}")
print("\n※ totalCnt 가 300 초과인 쿼리는 cap 때문에 최신 300건만 수집됨(선고일자 내림차순).")
print("※ 전체를 받으려면 수집 셀에서 MAX_PER_QUERY 를 상향하거나 None(전체) 으로.")

## 3. 법령해석례 (expc)

조문·개념 중심 키워드로 본문검색(`search=2`) → **쿼리별 카테고리 태깅** → `법령해석례일련번호` dedup →
상세 조회 → 2차 검증 → **카테고리별** `expc/expc_{카테고리}.jsonl` 저장.

> prec 와 동일 스킴(`EXPC_QUERY_CATEGORY` 쿼리→카테고리 매핑, 첫 매칭 카테고리 유지).
> 저장 위치는 타깃별 하위폴더 `data/01_raw/expc/` (eflaw/·prec/ 와 동일 구조).
> 이 셀은 자체 완결형이라 **설정 셀·헬퍼 셀만 실행돼 있으면 단독 재실행 가능**(eflaw/prec 재수집 불필요).

In [ ]:
from collections import defaultdict

# 쿼리 → 카테고리 매핑 (prec 의 QUERY_CATEGORY 와 동일 스킴)
EXPC_QUERY_CATEGORY: dict[str, str] = {
    "주택임대차보호법": "보증금권리", "임대차보증금 우선변제": "보증금권리",
    "대항력": "보증금권리", "확정일자": "보증금권리", "임차권등기명령": "보증금권리",
    "소액임차인 최우선변제": "보증금권리", "전입신고": "보증금권리",
    "계약갱신청구권": "갱신종료", "묵시적 갱신": "갱신종료", "차임 증액": "갱신종료",
    "상가건물 임대차": "상가임대차",
}

EXPC_QUERIES: list[str] = list(EXPC_QUERY_CATEGORY.keys())

# 저장 경로: 타깃별 하위폴더 (eflaw/, prec/ 와 동일 구조)
EXPC_DIR = RAW_DIR / "api_expc"
EXPC_DIR.mkdir(parents=True, exist_ok=True)

# 1) 본문검색 → 카테고리 태깅 → 합치기 → 법령해석례일련번호 1차 dedup
expc_items: list[dict] = []
for q in EXPC_QUERIES:
    hits = fetch_list("expc", query=q, search=2)
    cat = EXPC_QUERY_CATEGORY.get(q, "기타")
    for h in hits:
        h.setdefault("_category", cat)  # 첫 번째 매칭 카테고리만 유지
        h.setdefault("_query", q)
    print(f"[{q}] {len(hits)}건")
    expc_items.extend(hits)
    time.sleep(0.3)

before = len(expc_items)
expc_items = dedup_with_provenance(expc_items, "법령해석례일련번호")
print(f"\n수집 {before}건 → 법령해석례일련번호 1차 dedup {len(expc_items)}건")

# 2) 상세 조회 → 본문 유효성 필터 → 2차 dedup
expc_details = fetch_details("expc", expc_items, id_field="법령해석례일련번호")
expc_details = dedup_with_provenance(expc_details, "법령해석례일련번호")

# 3) 카테고리별 JSONL 저장
by_cat: dict[str, list] = defaultdict(list)
for rec in expc_details:
    by_cat[rec.get("_category", "기타")].append(rec)

for cat, recs in sorted(by_cat.items()):
    cat_path = EXPC_DIR / f"expc_{cat}.jsonl"
    with cat_path.open("w", encoding="utf-8") as f:
        for rec in recs:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print(f"저장 → {cat_path.name}  ({len(recs)}건)")

expc_md_count = write_api_markdown_files(
    "expc", expc_details, EXPC_DIR / "md", "법령해석례일련번호"
)
print(f"\n법령해석례 저장 완료: JSONL {len(expc_details)}건 + "
      f"MD {expc_md_count}개 → {EXPC_DIR}")

## 4. 수집 결과 요약 / 검증

In [ ]:
# 법령: 파일 수
eflaw_files = sorted(EFLAW_DIR.glob("*.json"))
print(f"eflaw: {len(eflaw_files)}개 파일 → {EFLAW_DIR}")

# 판례: 카테고리별 jsonl 합산 (api_prec/ 하위폴더)
PREC_DIR = globals().get("PREC_DIR") or (RAW_DIR / "api_prec")
prec_files = sorted(
    path for path in PREC_DIR.glob("prec_*.jsonl")
    if path.name != "prec_checkpoint.jsonl"
)
if prec_files:
    prec_recs = [json.loads(line) for p in prec_files for line in p.open(encoding="utf-8") if line.strip()]
    ids = [str(r.get("판례일련번호", "")) for r in prec_recs]
    leak = any("OC=" in json.dumps(r, ensure_ascii=False) for r in prec_recs)
    print(f"prec: {len(prec_recs)}건 ({len(prec_files)}개 카테고리) | ID 유일 {len(ids) == len(set(ids))} | OC 키 노출 {leak}")
else:
    print("prec: 파일 없음")

# 법령해석례: 카테고리별 jsonl 합산 (api_expc/ 하위폴더)
EXPC_DIR = globals().get("EXPC_DIR") or (RAW_DIR / "api_expc")
expc_files = sorted(EXPC_DIR.glob("expc_*.jsonl"))
if expc_files:
    expc_recs = [json.loads(line) for p in expc_files for line in p.open(encoding="utf-8") if line.strip()]
    ids = [str(r.get("법령해석례일련번호", "")) for r in expc_recs]
    leak = any("OC=" in json.dumps(r, ensure_ascii=False) for r in expc_recs)
    print(f"expc: {len(expc_recs)}건 ({len(expc_files)}개 카테고리) | ID 유일 {len(ids) == len(set(ids))} | OC 키 노출 {leak}")
    for p in expc_files:
        print(f"  - {p.name}: {sum(1 for _ in p.open(encoding='utf-8'))}건")
else:
    print("expc: 파일 없음")


## 5. 다음 단계: 전처리 노트북

이 노트북은 원본 수집까지만 담당한다. 저장된 원본은
`02_preprocess_legal_api.ipynb`에서 정제·metadata 생성 후
`03_build_kb_chunks.ipynb`에서 구조 기반으로 청킹한다.
